# 6.4.1 Vision-Language Models

Simulate CLIP-style cosine similarity between image and text embeddings (random numpy proxies) and compute InfoNCE contrastive loss.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

Path('output').mkdir(exist_ok=True)

def embed(seed, dim=64):
    rng = np.random.default_rng(seed)
    v = rng.standard_normal(dim)
    return v / (np.linalg.norm(v) + 1e-10)

def infonce_loss(sim, temp=0.07):
    n = sim.shape[0]
    s = sim / temp
    lse = np.log(np.sum(np.exp(s - s.max(1, keepdims=True)), 1)) + s.max(1)
    return (lse - s[np.arange(n), np.arange(n)]).mean()

# Matched image-text pairs (same seed = similar)
seeds = [10, 20, 30, 40, 50]
imgs = np.array([embed(s) for s in seeds])
txts = np.array([embed(s) for s in seeds])  # matched pairs
sim = imgs @ txts.T
loss = infonce_loss(sim)
print(f'Similarity matrix shape: {sim.shape}')
print(f'Diagonal (matched) mean: {np.diag(sim).mean():.3f}')
print(f'InfoNCE loss: {loss:.4f}')

In [ ]:
labels = ['dog', 'cat', 'car', 'plane', 'bird']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im = axes[0].imshow(sim, cmap='RdYlGn', vmin=-0.3, vmax=1)
axes[0].set(xticks=range(5), yticks=range(5),
            xticklabels=[f'text:{l}' for l in labels],
            yticklabels=[f'img:{l}' for l in labels],
            title='CLIP Cosine Similarity')
axes[0].tick_params(axis='x', rotation=30)
plt.colorbar(im, ax=axes[0])

temps = np.linspace(0.01, 0.5, 80)
losses = [infonce_loss(sim, t) for t in temps]
axes[1].plot(temps, losses, color='steelblue', lw=2)
axes[1].axvline(0.07, color='red', linestyle='--', label='τ=0.07')
axes[1].set(xlabel='Temperature τ', ylabel='InfoNCE Loss', title='Loss vs Temperature')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('output/vision_language.png')
print('Saved vision_language.png')